# 10 — Historical sampling-provenance audit

This notebook addresses Reviewer 1, Major Comment 1. It verifies the
observed composition of the frozen 400-study IUHN and 634-study MIMIC
cohorts, checks whether every paired file in each frozen test directory
entered the rerun, and inventories contemporaneous files that might
document the upstream selection. Balance is not treated as proof of
stratified sampling. If no historical selection record is found, the
output explicitly records that the original sampling rationale remains
unrecoverable from the supplied artifacts.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
sys.path[:] = [
    entry for entry in sys.path
    if Path(entry or ".").resolve() != implementation_dir
]
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
POST_ROOT = PATHS["root"] / "post_rerun"
POST_ROOT.mkdir(parents=True, exist_ok=True)
print("Code:", RERUN_DIR)
print("Output:", POST_ROOT)

In [ ]:
import hashlib, os, re
from datetime import datetime, timezone
import pandas as pd
from rerun_code.common import read_jsonl, write_json
from rerun_code.report_labeler import LABELS_13
from rerun_code.config import sha256_path

OUTPUT = POST_ROOT / "sampling_provenance"
OUTPUT.mkdir(parents=True, exist_ok=True)
composition_rows = []
directory_rows = []
frozen_manifests = {}

for dataset in ("mimic", "iuhn"):
    manifest_path = PATHS["manifests"] / dataset / "fixed_queries.jsonl"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Run Notebook 01 first: {manifest_path}")
    manifest = pd.DataFrame(read_jsonl(manifest_path))
    frozen_manifests[dataset] = manifest
    normal = manifest["labels_13"].map(len).eq(0)
    row = {
        "dataset": dataset,
        "n_studies": int(len(manifest)),
        "n_patient_or_uid_groups": int(manifest["patient_key"].nunique()),
        "patient_id_reliable": bool(manifest["patient_id_reliable"].all()),
        "n_normal": int(normal.sum()),
        "n_abnormal": int((~normal).sum()),
    }
    for label in LABELS_13:
        row[f"n_{label.replace(' ', '_')}"] = int(
            manifest["labels_13"].map(lambda labels: label in labels).sum()
        )
    composition_rows.append(row)

    test_root = Path(CONFIG["datasets"][dataset]["test"])
    if not test_root.exists():
        raise FileNotFoundError(test_root)
    image_suffixes = {".jpg", ".jpeg", ".png"}
    images = [p for p in test_root.rglob("*") if p.is_file() and p.suffix.lower() in image_suffixes]
    json_files = [p for p in test_root.rglob("*.json") if p.is_file()]
    image_keys = {str(p.relative_to(test_root).with_suffix("")) for p in images}
    json_keys = {str(p.relative_to(test_root).with_suffix("")) for p in json_files}
    directory_pairs = image_keys & json_keys
    manifest_image_names = set(manifest["image_path"].map(lambda value: Path(str(value)).name))
    directory_image_names = {p.name for p in images if str(p.relative_to(test_root).with_suffix("")) in directory_pairs}
    directory_rows.append({
        "dataset": dataset,
        "test_directory": str(test_root),
        "n_image_files": len(images),
        "n_json_files": len(json_files),
        "n_complete_image_json_pairs": len(directory_pairs),
        "n_frozen_manifest_records": len(manifest),
        "all_directory_pair_image_names_in_manifest": directory_image_names <= manifest_image_names,
        "all_manifest_image_names_in_directory_pairs": manifest_image_names <= directory_image_names,
        "manifest_sha256": sha256_path(manifest_path),
    })

composition = pd.DataFrame(composition_rows)
directory_audit = pd.DataFrame(directory_rows)
composition.to_csv(OUTPUT / "frozen_cohort_composition.csv", index=False)
directory_audit.to_csv(OUTPUT / "frozen_test_directory_audit.csv", index=False)
display(composition)
display(directory_audit)

In [ ]:
# Inventory likely provenance artifacts. This is an evidence inventory,
# not an automatic assertion about the historical selection method.
keywords = re.compile(r"sample|split|cohort|manifest|random|stratif|readme|build_config", re.I)
roots = [RERUN_DIR]
for dataset in ("mimic", "iuhn"):
    roots.extend([
        Path(CONFIG["datasets"][dataset]["test"]),
        Path(CONFIG["datasets"][dataset]["preencoded_test"]),
    ])
supplied = [Path(value) for value in os.environ.get("JAMIA_SAMPLING_EVIDENCE", "").split(",") if value.strip()]
roots.extend(path for path in supplied if path.is_dir())
candidates = {path.resolve() for path in supplied if path.is_file()}
for root in roots:
    if not root.exists() or not root.is_dir():
        continue
    root_depth = len(root.parts)
    for current, directories, files in os.walk(root):
        current_path = Path(current)
        if len(current_path.parts) - root_depth >= 4:
            directories[:] = []
        directories[:] = [name for name in directories if not name.startswith(".")]
        for name in files:
            path = current_path / name
            if keywords.search(name):
                candidates.add(path.resolve())

evidence_rows = []
for path in sorted(candidates):
    size = path.stat().st_size
    evidence_rows.append({
        "path": str(path),
        "size_bytes": size,
        "modified_utc": datetime.fromtimestamp(path.stat().st_mtime, timezone.utc).isoformat(),
        "sha256": sha256_path(path) if size <= 50 * 1024 * 1024 else "not_hashed_over_50MB",
        "explicitly_supplied": path in {item.resolve() for item in supplied if item.exists()},
        "automatic_sampling_claim_supported": False,
        "review_note": "Inspect manually; filename presence does not establish the sampling rule.",
    })
evidence = pd.DataFrame(evidence_rows)
evidence.to_csv(OUTPUT / "sampling_evidence_inventory.csv", index=False)

template_path = OUTPUT / "historical_sampling_provenance_template.csv"
pd.DataFrame([
    {
        "dataset": dataset,
        "source_population": "",
        "eligibility_criteria": "",
        "selection_method": "",
        "stratification_variables": "",
        "random_seed": "",
        "selection_date_or_version": "",
        "responsible_investigator": "",
        "evidence_file": "",
        "manuscript_wording": "",
    }
    for dataset in ("mimic", "iuhn")
]).to_csv(template_path, index=False)

completed_path = Path(os.environ.get(
    "JAMIA_COMPLETED_SAMPLING_PROVENANCE",
    OUTPUT / "historical_sampling_provenance_completed.csv",
))
historical_resolved = False
validation_errors = []
if completed_path.exists():
    completed = pd.read_csv(completed_path, dtype=str).fillna("")
    required = [
        "dataset", "source_population", "eligibility_criteria", "selection_method",
        "evidence_file", "manuscript_wording",
    ]
    missing = [name for name in required if name not in completed]
    if missing:
        validation_errors.append(f"missing columns: {missing}")
    if not missing:
        if set(completed["dataset"]) != {"mimic", "iuhn"}:
            validation_errors.append("completed file must contain one mimic row and one iuhn row")
        for name in required[1:]:
            if completed[name].str.strip().eq("").any():
                validation_errors.append(f"blank required values in {name}")
        for value in completed["evidence_file"]:
            if value.strip() and not Path(value).exists():
                validation_errors.append(f"evidence file not found: {value}")
    historical_resolved = not validation_errors

status = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "observed_cohort_composition_verified": True,
    "all_frozen_test_directory_pairs_analyzed": bool(
        directory_audit["all_directory_pair_image_names_in_manifest"].all()
        and directory_audit["all_manifest_image_names_in_directory_pairs"].all()
    ),
    "historical_sampling_provenance_resolved": historical_resolved,
    "historical_sampling_provenance_status": (
        "documented from completed evidence file"
        if historical_resolved else
        "not established by current computational artifacts; complete the provenance template from contemporaneous records"
    ),
    "validation_errors": validation_errors,
    "completed_provenance_file": str(completed_path),
    "outputs": {
        "cohort_composition": str(OUTPUT / "frozen_cohort_composition.csv"),
        "test_directory_audit": str(OUTPUT / "frozen_test_directory_audit.csv"),
        "evidence_inventory": str(OUTPUT / "sampling_evidence_inventory.csv"),
        "provenance_template": str(template_path),
    },
    "interpretation_guardrail": "Observed class balance is not proof of stratified sampling.",
}
write_json(OUTPUT / "notebook10_status.json", status)
print(json.dumps(status, indent=2))